# Week 6 PySpark Lab: DataFrames + Delta + Medallion

This notebook runs on a **local PySpark + Delta Lake** Docker container (see `environment/pyspark/`).
It covers the same concepts as the Databricks exercises: DataFrames, medallion architecture, and Delta history/time travel.

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Week6Lab")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} — Delta ready")

Spark 3.5.0 — Delta ready


## 1) Create a DataFrame and run core operations

In [2]:
data = [
    (1, "C01", "Laptop",   120.50),
    (2, "C02", "Mouse",     25.00),
    (3, "C01", "Keyboard",  70.00),
    (4, "C03", "Monitor",  220.00),
]

df = spark.createDataFrame(data, ["order_id", "customer_id", "product", "amount"])
df.show()

+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       1|        C01|  Laptop| 120.5|
|       2|        C02|   Mouse|  25.0|
|       3|        C01|Keyboard|  70.0|
|       4|        C03| Monitor| 220.0|
+--------+-----------+--------+------+



In [3]:
high_value = df.filter(F.col("amount") > 50)
by_customer = df.groupBy("customer_id").agg(F.sum("amount").alias("total_spend"))

print("--- Orders with amount > 50 ---")
high_value.show()
print("--- Spend by customer (descending) ---")
by_customer.orderBy(F.col("total_spend").desc()).show()

--- Orders with amount > 50 ---
+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       1|        C01|  Laptop| 120.5|
|       3|        C01|Keyboard|  70.0|
|       4|        C03| Monitor| 220.0|
+--------+-----------+--------+------+

--- Spend by customer (descending) ---
+-----------+-----------+
|customer_id|total_spend|
+-----------+-----------+
|        C03|      220.0|
|        C01|      190.5|
|        C02|       25.0|
+-----------+-----------+



## 2) Bronze -> Silver -> Gold (Medallion pattern)

In [4]:
DELTA_BASE = "/home/jovyan/work/delta"

# Bronze: raw records as-is
df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/bronze_orders")
print("Bronze written")

# Silver: cast amount to DECIMAL, deduplicate
silver_df = (
    spark.read.format("delta").load(f"{DELTA_BASE}/bronze_orders")
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .dropDuplicates(["order_id"])
)
silver_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/silver_orders")
print("Silver written")

# Gold: aggregated revenue per customer
gold_df = (
    spark.read.format("delta").load(f"{DELTA_BASE}/silver_orders")
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total_revenue"))
)
gold_df.write.format("delta").mode("overwrite").save(f"{DELTA_BASE}/gold_customer_revenue")
print("Gold written")

print("\n--- Gold: revenue per customer ---")
spark.read.format("delta").load(f"{DELTA_BASE}/gold_customer_revenue").orderBy("customer_id").show()

Bronze written
Silver written
Gold written

--- Gold: revenue per customer ---
+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|        C01|       190.50|
|        C02|        25.00|
|        C03|       220.00|
+-----------+-------------+



## 3) Delta history and time travel

In [5]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, f"{DELTA_BASE}/silver_orders")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                           |
+-------+-----------------------+---------+-----------------------------------------------------------+
|0      |2026-05-29 18:16:39.921|WRITE    |{numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1359}|
+-------+-----------------------+---------+-----------------------------------------------------------+



In [6]:
# Create version 1: update one record's amount
dt.update(condition="order_id = 1", set={"amount": F.lit(121.50)})

print("--- History after update ---")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

--- History after update ---
+-------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                                                                            |
+-------+-----------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
# Version 0: original write — order_id=1 amount is still 120.50
print("--- VERSION AS OF 0 (original write) ---")
spark.read.format("delta").option("versionAsOf", 0).load(f"{DELTA_BASE}/silver_orders").orderBy("order_id").show()

print("--- Current version (after UPDATE) ---")
spark.read.format("delta").load(f"{DELTA_BASE}/silver_orders").orderBy("order_id").show()

--- VERSION AS OF 0 (original write) ---
+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       1|        C01|  Laptop|120.50|
|       2|        C02|   Mouse| 25.00|
|       3|        C01|Keyboard| 70.00|
|       4|        C03| Monitor|220.00|
+--------+-----------+--------+------+

--- Current version (after UPDATE) ---
+--------+-----------+--------+------+
|order_id|customer_id| product|amount|
+--------+-----------+--------+------+
|       1|        C01|  Laptop|121.50|
|       2|        C02|   Mouse| 25.00|
|       3|        C01|Keyboard| 70.00|
|       4|        C03| Monitor|220.00|
+--------+-----------+--------+------+



## 4) Quick checks

- `df.show()` returns 4 rows (order_id 1–4).
- `high_value` filters to 3 rows (amount > 50: Laptop, Keyboard, Monitor).
- Bronze, Silver, Gold Delta tables written to `/home/jovyan/work/delta/`.
- Delta history shows version `0` (WRITE) then version `1` (UPDATE).
- Time travel `VERSION AS OF 0` shows `amount = 120.50` for order_id 1; current shows `121.50`.